In [ ]:
import time

import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier

pd.set_option("display.max_rows", None)  # or a specific number
pd.set_option("display.max_columns", None)  # to show all columns
pd.set_option("display.expand_frame_repr", False)  # to allow wider DataFrame display

In [4]:
features = pd.read_csv("features/features.csv", index_col='match_id')

In [5]:
target_col = "radiant_win"
features_to_remove = [
    "duration",
    "tower_status_radiant",
    "tower_status_dire",
    "barracks_status_dire",
    "barracks_status_radiant",
    "start_time",
    "lobby_type",
]
y = features[target_col].copy()
X = features.drop(features_to_remove + [target_col], axis=1)
X = X.fillna(0)

In [6]:
model = CatBoostClassifier(
    n_estimators=30,
    cat_features=[
        "r1_hero",
        "r2_hero",
        "r3_hero",
        "r4_hero",
        "r5_hero",
        "d1_hero",
        "d2_hero",
        "d3_hero",
        "d4_hero",
        "d5_hero",
    ],  # Specify categorical features
    random_seed=42,
)

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=241)
for train_ind, test_ind in kf.split(X, y):
    X_train, X_test = X.iloc[train_ind], X.iloc[test_ind]
    y_train, y_test = y.iloc[train_ind], y.iloc[test_ind]

    start_time = time.time()
    model.fit(X_train, y_train, verbose=0)
    elapsed_time = time.time() - start_time

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_pred_proba)

    print(f"Fold AUC-ROC: {score:.4f} | Time taken: {elapsed_time:.2f} sec")

Fold AUC-ROC: 0.7203 | Time taken: 1.32 sec
Fold AUC-ROC: 0.7114 | Time taken: 1.08 sec
Fold AUC-ROC: 0.7183 | Time taken: 1.52 sec
Fold AUC-ROC: 0.7162 | Time taken: 1.22 sec
Fold AUC-ROC: 0.7244 | Time taken: 1.05 sec
